In [2]:
%load_ext autoreload

In [3]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.pyplot import figure

%autoreload 2
%aimport AD_predictor_tools
%aimport AD_comparison_tools
%aimport PlottingTools

In [9]:
SFARI_Gene = pd.read_csv("../data/SFARI-Gene_genes_01-23-2023release_03-21-2023export.csv")
SFARI_Gene

,status,gene-symbol,gene-name,ensembl-id,chromosome,genetic-category,gene-score,syndromic,eagle,number-of-reports
0,9,ABAT,4-aminobutyrate aminotransferase,ENSG00000183044,16,"Rare Single Gene Mutation, Genetic Association",2.0,0,NaN,7
1,9,ABCA10,"ATP-binding cassette, sub-family A (ABC1), mem...",ENSG00000154263,17,Rare Single Gene Mutation,2.0,0,NaN,2
2,9,ABCA13,ATP binding cassette subfamily A member 13,ENSG00000179869,7,"Rare Single Gene Mutation, Functional",2.0,0,NaN,10
3,9,ABCA7,"ATP-binding cassette, sub-family A (ABC1), mem...",ENSG00000064687,19,Rare Single Gene Mutation,2.0,0,NaN,5
4,9,ABL2,"ABL proto-oncogene 2, non-receptor tyrosine ki...",ENSG00000143322,1,"Rare Single Gene Mutation, Functional",3.0,0,NaN,10
...,...,...,...,...,...,...,...,...,...,...
1113,9,ZNF774,Zinc finger protein 774,ENSG00000196391,15,Rare Single Gene Mutation,2.0,0,NaN,4
1114,9,ZNF804A,Zinc finger protein 804A,ENSG00000170396,2,"Rare Single Gene Mutation, Genetic Association...",2.0,0,NaN,16
1115,9,ZNF827,Zinc finger protein 827,ENSG00000151612,4,"Rare Single Gene Mutation, Genetic Association",2.0,0,NaN,3
1116,9,ZSWIM6,zinc finger SWIM-type containing 6,ENSG00000130449,5,"Rare Single Gene Mutation, Syndromic, Genetic ...",NaN,1,NaN,6


In [36]:
SFARI_Gene[SFARI_Gene["ensembl-id"].isna()]

,status,gene-symbol,gene-name,ensembl-id,chromosome,genetic-category,gene-score,syndromic,eagle,number-of-reports
11,9,ADA,adenosine deaminase,NaN,20,"Rare Single Gene Mutation, Genetic Association",2.0,0,NaN,8
66,9,AR,androgen receptor,NaN,X,Genetic Association,2.0,0,NaN,6
559,9,MAPT-AS1,MAPT antisense RNA 1,NaN,17,Genetic Association,2.0,0,NaN,1
583,9,MET,met proto-oncogene (hepatocyte growth factor r...,NaN,7,"Rare Single Gene Mutation, Genetic Association...",2.0,0,NaN,40
588,9,MIR137,microRNA 137,NaN,1,"Rare Single Gene Mutation, Syndromic, Genetic ...",2.0,0,NaN,10
631,9,MSNP1AS,"Moesinpseudogene 1, antisense",NaN,5,"Genetic Association, Functional",2.0,0,NaN,13
792,9,PTCHD1-AS,PTCHD1antisense RNA (head to head),NaN,X,Rare Single Gene Mutation,2.0,0,17.6,3
847,9,RP11-1407O15.2,NaN,NaN,17,Rare Single Gene Mutation,2.0,0,NaN,1
871,9,RPS10P2-AS1,ribosomal protein S10 pseudogene 2 anti-sense 1,NaN,20,"Genetic Association, Functional",2.0,0,NaN,4


In [6]:
# Redownloading lambert TFs from uniprot.

# First, downloaded tables s1-s4 from lambert review
lambert_table_s1 = pd.read_excel(open('../data/lambert_supp_tables.xlsx', 'rb'),
                               sheet_name='Table S1. Related to Figure 1B')
# Keeping TF rows
lambert_table_s1_TF_rows = lambert_table_s1[lambert_table_s1["Is TF?"] == "Yes"]
lambert_table_s1_TF_rows

,Gene Information,Unnamed: 1,Unnamed: 2,Is TF?,Final Assesment,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Notes from re-reviewed genes,...,Unnamed: 22,Unnamed: 23,Unnamed: 24,Unnamed: 25,Prior classifications,Unnamed: 27,Unnamed: 28,Unnamed: 29,Unnamed: 30,Unnamed: 31
1,ENSG00000137203,TFAP2A,AP-2,Yes,Known motif,1 Monomer or homomultimer,High-throughput in vitro,NaN,NaN,NaN,...,Has known motif,1 Monomer or homomultimer,NaN,NaN,a,Yes,Yes,TF Gene_DNA-Binding: sequence-specific_DNA Bin...,Yes,NaN
2,ENSG00000008196,TFAP2B,AP-2,Yes,Known motif,1 Monomer or homomultimer,High-throughput in vitro,NaN,NaN,NaN,...,Has known motif,1 Monomer or homomultimer,NaN,NaN,a,Yes,Yes,TF Gene_DNA-Binding: sequence-specific_DNA Bin...,Yes,NaN
3,ENSG00000087510,TFAP2C,AP-2,Yes,Known motif,1 Monomer or homomultimer,High-throughput in vitro,NaN,NaN,NaN,...,Has known motif,1 Monomer or homomultimer,NaN,NaN,a,Yes,Yes,No,Yes,NaN
4,ENSG00000008197,TFAP2D,AP-2,Yes,Known motif,1 Monomer or homomultimer,In vivo/Misc source,Only known motifs are from Transfac or HocoMoc...,Binds the same GCCTGAGGC sequence as the other...,NaN,...,Has known motif,1 Monomer or homomultimer,Source of Hocomoco motif is unclear,NaN,a,Yes,Yes,No,Yes,NaN
5,ENSG00000116819,TFAP2E,AP-2,Yes,Known motif,1 Monomer or homomultimer,High-throughput in vitro,NaN,NaN,NaN,...,Has known motif,1 Monomer or homomultimer,NaN,NaN,a,Yes,Yes,TF Gene_DNA-Binding: sequence-specific_DNA Bin...,Yes,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2760,ENSG00000177683,THAP5,THAP finger,Yes,Likely to be sequence specific TF,1 Monomer or homomultimer,No motif,NaN,NaN,NaN,...,Likely to be sequence specific TF,1 Monomer or homomultimer,NaN,NaN,c,Yes,Yes,No,No,NaN
2761,ENSG00000174796,THAP6,THAP finger,Yes,Likely to be sequence specific TF,1 Monomer or homomultimer,No motif,NaN,NaN,NaN,...,Likely to be sequence specific TF,1 Monomer or homomultimer,NaN,NaN,c,Yes,Yes,No,No,NaN
2762,ENSG00000184436,THAP7,THAP finger,Yes,Likely to be sequence specific TF,1 Monomer or homomultimer,No motif,NaN,NaN,NaN,...,Likely to be sequence specific TF,1 Monomer or homomultimer,NaN,NaN,c,Yes,Yes,No,No,NaN
2763,ENSG00000161277,THAP8,THAP finger,Yes,Likely to be sequence specific TF,1 Monomer or homomultimer,No motif,NaN,NaN,NaN,...,Likely to be sequence specific TF,NaN,PMID: 12575992 says it has unique DBD,need revisit as it has C2CH signature,c,Yes,Yes,No,No,NaN


In [7]:
lambert_TF_ensg_codes = lambert_table_s1_TF_rows["Gene Information"]
lambert_TF_ensg_codes

1       ENSG00000137203
2       ENSG00000008196
3       ENSG00000087510
4       ENSG00000008197
5       ENSG00000116819
             ...       
2760    ENSG00000177683
2761    ENSG00000174796
2762    ENSG00000184436
2763    ENSG00000161277
2764    ENSG00000168152
Name: Gene Information, Length: 1639, dtype: object

In [10]:
# Rows of SFARI Gene with ENSG codes in lambert TFs
SFARI_TFs = SFARI_Gene[SFARI_Gene["ensembl-id"].isin(lambert_TF_ensg_codes)]
SFARI_TFs

,status,gene-symbol,gene-name,ensembl-id,chromosome,genetic-category,gene-score,syndromic,eagle,number-of-reports
16,9,ADNP,Activity-dependent neuroprotector homeobox,ENSG00000101126,20,"Rare Single Gene Mutation, Syndromic, Functional",1.0,1,41.50,64
31,9,AHDC1,AT-hook DNA binding motif containing 1,ENSG00000126705,1,"Rare Single Gene Mutation, Syndromic",1.0,1,14.25,24
60,9,ARID2,AT-rich interaction domain 2,ENSG00000189079,12,"Rare Single Gene Mutation, Syndromic",2.0,1,NaN,12
61,9,ARNT2,aryl-hydrocarbon receptor nuclear translocator 2,ENSG00000172379,15,"Rare Single Gene Mutation, Syndromic, Genetic ...",2.0,0,NaN,15
62,9,ARX,aristaless related homeobox,ENSG00000004848,X,"Rare Single Gene Mutation, Syndromic",1.0,1,13.80,24
...,...,...,...,...,...,...,...,...,...,...
1111,9,ZNF711,zinc finger protein 711,ENSG00000147180,X,Rare Single Gene Mutation,2.0,0,NaN,5
1112,9,ZNF713,Zinc finger protein 713,ENSG00000178665,7,Rare Single Gene Mutation,2.0,0,NaN,3
1113,9,ZNF774,Zinc finger protein 774,ENSG00000196391,15,Rare Single Gene Mutation,2.0,0,NaN,4
1114,9,ZNF804A,Zinc finger protein 804A,ENSG00000170396,2,"Rare Single Gene Mutation, Genetic Association...",2.0,0,NaN,16


In [11]:
import numpy as np
from scipy.stats import hypergeom
import matplotlib.pyplot as plt

In [21]:
k = 1118 ## Number of SFARI Genes
m = 1639 ## Number of TFs
N = 20000 ## Total number of genes 
n = N - m ## Number of non TF genes
x = 126 ## Number of SFARI TFs

# Python
hypergeom(M=N, 
                n=m, 
                N=k).sf(x-1)

0.00013579589468489987

In [26]:
SFARI_gene_category_counts = dict(SFARI_Gene["gene-score"].value_counts())
SFARI_gene_category_counts

{2.0: 699, 1.0: 232, 3.0: 97}

In [27]:
SFARI_TFs_cateogry_counts = dict(SFARI_TFs["gene-score"].value_counts())
SFARI_TFs_cateogry_counts

{2.0: 57, 1.0: 51, 3.0: 8}

In [32]:
def return_p_val(category):
    k = SFARI_gene_category_counts[category] ## Number of SFARI category Genes
    m = 1639 ## Number of TFs
    N = 20000 ## Total number of genes 
    n = N - m ## Number of non TF genes
    x = SFARI_TFs_cateogry_counts[category] ## Number of SFARI TFs

    print("There are " + str(x) + " TFs out of " + str(k) + " Category " + str(category) + " Genes")
    
    # Python
    print(hypergeom(M=N, 
                    n=m, 
                    N=k).sf(x-1))

In [34]:
return_p_val(1)

There are 51 TFs out of 232 Category 1 Genes
5.610522037071283e-11


In [33]:
return_p_val(2)

There are 57 TFs out of 699 Category 2 Genes
0.5365914889839618


In [35]:
return_p_val(3)

There are 8 TFs out of 97 Category 3 Genes
0.5459567503411658


In [41]:
51/232

0.21982758620689655

In [38]:
8/ 97

0.08247422680412371

In [39]:
57/699

0.0815450643776824

In [40]:
1639 / 20000

0.08195

In [ ]:
hypergeom.sf(x-1, M = 20000, n = 126, N = 1118)